# UC06 — Análisis NLP de Incidencias y Partes de Trabajo

**Objetivo:** Clasificar y extraer información de partes de trabajo en texto libre para detectar patrones y priorizar inversiones.

**Tecnologías:** CORTEX.COMPLETE · CORTEX.SENTIMENT · Streamlit

**Tiempo estimado:** 11 minutos

## Paso 1 — Configuración del Entorno

In [ ]:
CREATE DATABASE IF NOT EXISTS INCIDENCIAS_NLP_DB;

In [ ]:
USE DATABASE INCIDENCIAS_NLP_DB;

In [ ]:
USE SCHEMA PUBLIC;

In [ ]:
CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH WITH WAREHOUSE_SIZE='SMALL' AUTO_SUSPEND=60 AUTO_RESUME=TRUE;

In [ ]:
USE WAREHOUSE COMPUTE_WH;

## Paso 2 — Generar Partes de Trabajo

500 partes con texto libre simulando observaciones de operarios de campo.

In [ ]:
CREATE OR REPLACE TABLE partes_trabajo AS
SELECT
    'PT-' || LPAD(SEQ4()::STRING, 5, '0') AS parte_id,
    DATEADD('day', -UNIFORM(0, 365, RANDOM()), CURRENT_DATE()) AS fecha,
    'OP-' || LPAD(MOD(SEQ4(), 20)::STRING, 3, '0') AS operario_id,
    CASE MOD(SEQ4(), 5) WHEN 0 THEN 'Red_Norte' WHEN 1 THEN 'Red_Sur' WHEN 2 THEN 'ETAP' WHEN 3 THEN 'EDAR' ELSE 'Depositos' END AS zona,
    CASE MOD(SEQ4(), 5) WHEN 0 THEN 'Averia' WHEN 1 THEN 'Fuga' WHEN 2 THEN 'Calidad' WHEN 3 THEN 'Vandalismo' ELSE 'Obstruccion' END AS tipo_real,
    CASE MOD(SEQ4(), 5)
        WHEN 0 THEN 'Se detecta avería en bomba de impulsión ' || MOD(SEQ4(), 10) || '. Vibraciones elevadas y ruido anormal en rodamiento. Se procede a parada y solicitud de repuesto. Equipo fuera de servicio hasta reparación.'
        WHEN 1 THEN 'Fuga visible en tubería de distribución calle ' || MOD(SEQ4(), 50) || '. Rotura en junta de unión. Caudal estimado de pérdida: ' || (5 + MOD(SEQ4(), 20)) || ' m3/h. Se procede a corte y reparación urgente.'
        WHEN 2 THEN 'Reclamación de vecinos por agua turbia en zona ' || MOD(SEQ4(), 20) || '. Se toman muestras in situ: turbidez ' || (2 + MOD(SEQ4(), 15)) || ' NTU, cloro residual ' || ROUND(0.1 + MOD(SEQ4(), 8) * 0.1, 1) || ' mg/l. Se programa purga de red.'
        WHEN 3 THEN 'Acto vandálico en hidrante/arqueta zona ' || MOD(SEQ4(), 30) || '. Tapa de registro arrancada y contador manipulado. Se documenta fotográficamente y se presenta denuncia. Reparación inmediata.'
        ELSE 'Obstrucción en colector principal sector ' || MOD(SEQ4(), 15) || '. Acumulación de grasas y residuos sólidos. Se realiza limpieza con equipo de alta presión. Recomendación: instalar trampa de grasas.'
    END AS texto_observaciones,
    CASE MOD(SEQ4(), 4) WHEN 0 THEN 'Baja' WHEN 1 THEN 'Media' WHEN 2 THEN 'Alta' ELSE 'Critica' END AS urgencia_real
FROM TABLE(GENERATOR(ROWCOUNT => 500));

In [ ]:
SELECT tipo_real, COUNT(*) AS partes, zona FROM partes_trabajo GROUP BY tipo_real, zona ORDER BY 2 DESC LIMIT 15;

## Paso 3 — Clasificar Incidencias con IA

Usamos CORTEX.COMPLETE para categorizar cada parte de trabajo automáticamente.

In [ ]:
CREATE OR REPLACE TABLE partes_clasificados AS
SELECT *,
    SNOWFLAKE.CORTEX.COMPLETE('mistral-large2',
        'Clasifica el siguiente parte de trabajo en UNA de estas categorías: Averia, Fuga, Calidad, Vandalismo, Obstruccion, Otro. Responde SOLO con la categoría. Texto: ' || texto_observaciones
    )::STRING AS tipo_predicho,
    SNOWFLAKE.CORTEX.SENTIMENT(texto_observaciones) AS sentimiento_score
FROM partes_trabajo;

In [ ]:
SELECT tipo_real, tipo_predicho, COUNT(*) AS n FROM partes_clasificados GROUP BY 1, 2 ORDER BY 1, 3 DESC;

## Paso 4 — Detectar Patrones Recurrentes

Analizamos qué zonas y equipos concentran más incidencias.

In [ ]:
SELECT zona, tipo_real, COUNT(*) AS incidencias, ROUND(AVG(sentimiento_score), 2) AS sentimiento_medio
FROM partes_clasificados GROUP BY zona, tipo_real ORDER BY incidencias DESC LIMIT 20;

## Paso 5 — Dashboard de Incidencias

In [ ]:
import streamlit as st
from snowflake.snowpark.context import get_active_session
session = get_active_session()

st.title("📋 Análisis NLP de Incidencias")

col1, col2, col3 = st.columns(3)
total = session.sql("SELECT COUNT(*) AS n FROM partes_clasificados").collect()[0]["N"]
col1.metric("Partes Analizados", total)
col2.metric("Zonas", 5)
col3.metric("Tipos de Incidencia", 5)

st.subheader("Distribución por Tipo y Zona")
df = session.sql("SELECT zona, tipo_real, COUNT(*) AS n FROM partes_clasificados GROUP BY 1,2 ORDER BY 3 DESC").to_pandas()
st.dataframe(df, use_container_width=True)

## Paso 6 — Limpieza (Opcional)

In [ ]:
-- DROP DATABASE IF EXISTS INCIDENCIAS_NLP_DB;